# EcoRoute — Model Evaluation Notebook

This notebook reproduces the full ML evaluation pipeline:
1. Load the 50k synthetic delivery dataset
2. Inspect feature distributions and correlations
3. Train all 6 candidate models
4. Compare metrics (MAE, RMSE, R², MAPE, CV-R²)
5. Visualise predictions vs actuals
6. Inspect feature importance of the winning model
7. Save the comparison report

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml.features import load_dataset, prepare_features, add_cyclical_features
from ml.train import evaluate_model, compute_metrics
from ml.model_registry import build_model, list_available_models

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Available models:', list_available_models())

## 1. Load dataset

In [ ]:
df = load_dataset('../data/raw/deliveries.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Feature distributions

In [ ]:
numeric_cols = ['distance_km', 'load_weight_kg', 'fuel_used_l', 'delivery_time_h',
                'co2_emission_kg', 'temperature_c', 'rainfall_mm', 'wind_speed_kmh']
df[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sns.histplot(df['fuel_used_l'], bins=50, ax=axes[0,0], color='#22c55e')
axes[0,0].set_title('Fuel Used Distribution')
sns.histplot(df['distance_km'], bins=50, ax=axes[0,1], color='#0ea5e9')
axes[0,1].set_title('Distance Distribution')
sns.boxplot(data=df, x='vehicle_type', y='fuel_used_l', ax=axes[1,0])
axes[1,0].set_title('Fuel by Vehicle Type')
axes[1,0].tick_params(axis='x', rotation=45)
sns.boxplot(data=df, x='traffic_level', y='fuel_used_l', ax=axes[1,1])
axes[1,1].set_title('Fuel by Traffic Level')
plt.tight_layout()
plt.show()

## 3. Correlation heatmap

In [ ]:
df_cyc = add_cyclical_features(df)
numeric_df = df_cyc.select_dtypes(include=[np.number])
plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdYlGn_r', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Train all 6 models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from ml.features import build_preprocessor

X, y = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
predictions = {}

CV_SKIP = {'gradient_boosting', 'catboost'}  # memory-constrained envs
for name in list_available_models():
    print(f'\n=== Training {name} ===')
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', build_model(name))])
    m = evaluate_model(name, pipe, X_train, y_train, X_test, y_test,
                       skip_cv=name in CV_SKIP)
    results[name] = m
    predictions[name] = pipe.predict(X_test)

## 5. Comparison table

In [ ]:
comparison = pd.DataFrame(results).T
comparison = comparison.sort_values('r2', ascending=False)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
comparison[['mae', 'rmse']].plot(kind='bar', ax=axes[0], color=['#f59e0b', '#ef4444'])
axes[0].set_title('MAE / RMSE (lower is better)')
axes[0].set_ylabel('Litres')
axes[0].tick_params(axis='x', rotation=30)

comparison['r2'].plot(kind='bar', ax=axes[1], color='#22c55e')
axes[1].set_title('R² Score (higher is better)')
axes[1].set_ylabel('R²')
axes[1].set_ylim(0.8, 1.0)
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 6. Predictions vs actuals (best model)

In [ ]:
best_name = comparison.index[0]
y_pred = predictions[best_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred, alpha=0.3, s=8, c='#22c55e')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual Fuel (L)')
axes[0].set_ylabel('Predicted Fuel (L)')
axes[0].set_title(f'{best_name}: Predicted vs Actual')

residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, color='#0ea5e9', edgecolor='white')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual (L)')
plt.tight_layout()
plt.show()

## 7. Feature importance (best tree model)

In [ ]:
if best_name in ('random_forest', 'xgboost', 'lightgbm', 'catboost'):
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', build_model(best_name))])
    pipe.fit(X_train, y_train)
    model = pipe.named_steps['model']
    
    # Get feature names from preprocessor
    pre = pipe.named_steps['preprocessor']
    num_features = pre.transformers_[0][2]
    cat_encoder = pre.transformers_[1][1].named_steps['onehot']
    cat_features = cat_encoder.get_feature_names_out(pre.transformers_[1][2]).tolist()
    all_features = num_features + cat_features
    
    importances = model.feature_importances_ if hasattr(model, 'feature_importances_') else None
    if importances is not None:
        imp_df = pd.DataFrame({'feature': all_features, 'importance': importances})
        imp_df = imp_df.sort_values('importance', ascending=False).head(15)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
        plt.title(f'Top 15 Feature Importances ({best_name})')
        plt.tight_layout()
        plt.show()

## 8. Save report

In [ ]:
import json
from pathlib import Path

report = {
    'best_model': best_name,
    'best_metrics': results[best_name],
    'all_results': results,
}
Path('../ml/reports').mkdir(parents=True, exist_ok=True)
with open('../ml/reports/notebook_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f'Report saved. Best model: {best_name} (R²={results[best_name]["r2"]})')